This notebook loads the pickled results produced by `scaling_effdim_dimin.py` and plots how the normalized effective dimension (ED, Eq. 12) scales with the input-basis (correlation-space) dimension D (`dim_in`, shown as $D/M$), at a fixed target correlation-spectrum purity $\mathrm{tr}\,S^4=$ `purity0`, for a few fixed (M, Dloc) architectures.

Import the plotting and numerical packages, and add the `useful_functions` folder to the module search path so the tensor-network, FIM, and model-construction helper modules used by the generating script can be (re)imported.

In [ ]:
# Importing necessary packages
import sys
import os
import importlib
import pickle
import numpy

import pennylane.numpy as np




import matplotlib.pyplot as plt


# Current path for importing custom functions
path_base = '/home/b/b309245/FIM_Training_Bias_RegressionModels/fourier_models_training_and_fim/'
sys.path.insert(0, path_base + 'useful_functions')

import model_constructor_functions
importlib.reload(model_constructor_functions)

import ortho_matrices_functions
importlib.reload(ortho_matrices_functions)

import tensor_network_functions_np
importlib.reload(tensor_network_functions_np)

import FIM_functions_jax
importlib.reload(FIM_functions_jax)

import training_functions_jax
importlib.reload(training_functions_jax)

Set the results folder and the fixed parameters shared by all experiments, including the target correlation-spectrum purity $\mathrm{tr}\,S^4$ (`purity0`) held fixed while D is scanned, then list the paired values of Dloc (`local_dim_param_vec`) and M (`no_params_vec`) for each experiment/curve to load.

In [ ]:
results_folder = '/work/bd1179/b309245/fourier_models_train_and_FIM/scaling_effdim_TN_dimInSpace/'

# Fixed purity of S
purity0 = 0.333
purity0_name = '0p333'

# No. of random parameter samples for evaluating normalized eff. dim.
no_samples = 200
no_par_samples_name = str(no_samples)

# No. of random orthogonal matrix realizations per S decay exponent
no_matrix_realiz = 30
no_V_samples_name = str(no_matrix_realiz)

# Vector of bond dimensions
bond_dim = 120
name_bond_dim = str(bond_dim)

########################### Exps. to plot ###########################

# Local dimension of parameter functions space
local_dim_param_vec = [5, 7]
name_dim_par_loc_vec = [str(local_dim_param) for local_dim_param in local_dim_param_vec]

# Dimension of input function space
no_params_vec = [50, 80]
name_no_params_vec = [str(no_params) for no_params in no_params_vec]

no_exps = len(local_dim_param_vec)

For each experiment (each Dloc/M pair), find and load all pickled `norm_eff_dim_*` files matching that architecture and target purity, concatenating the swept D values, correlation-spectrum purity $\mathrm{tr}\,S^4$, and normalized ED arrays across all random model realizations into a per-experiment dictionary `dict_exps[i]`.

In [ ]:
dict_exps = dict()

for i in range(no_exps):
    local_dim_param = local_dim_param_vec[i]
    name_dim_par_loc = name_dim_par_loc_vec[i]
    no_params = no_params_vec[i]
    name_no_params = name_no_params_vec[i]

    namefileend = ('_BondDim' + name_bond_dim + '_Nparams' + name_no_params + '_DimLocPar' + name_dim_par_loc + 
                   '_S4trace' + purity0_name  + '_NsamplesPar' + no_par_samples_name + 
                   '_NsamplesV' + no_V_samples_name + '_DimIn')
    
    namefileini = 'norm_eff_dim' + namefileend
    listallfiles = [f for f in os.listdir(results_folder) if (f.startswith(namefileini))]
    print(len(listallfiles))
    
    dim_in_all = []
    purity_S_all = []
    norm_eff_dim_all = []
    
    for filename in listallfiles:
        path_file = os.path.join(results_folder, filename)
        with open(path_file, 'rb') as f:
            dict_norm_ed = pickle.load(f)
    
        dimin_vec = dict_norm_ed['dim_in_all']
        purity_S_vec = dict_norm_ed['purity_S_all']
        norm_eff_dim_vec = dict_norm_ed['norm_eff_dim_all']
    
        dim_in_all.append(dimin_vec)
        purity_S_all.append(purity_S_vec)
        norm_eff_dim_all.append(norm_eff_dim_vec)
    
    dim_in_all = np.concatenate(dim_in_all)
    purity_S_all = np.concatenate(purity_S_all)
    norm_eff_dim_all = np.concatenate(norm_eff_dim_all)

    dict_exp = dict()
    dict_exp['local_dim_param'] = local_dim_param
    dict_exp['no_params'] = no_params
    dict_exp['purity_S_all'] = purity_S_all
    dict_exp['dim_in_all'] = dim_in_all
    dict_exp['norm_eff_dim_all'] = norm_eff_dim_all
    dict_exps[i] = dict_exp

Compute the mean correlation-spectrum purity $\mathrm{tr}\,S^4$ for each experiment and plot the normalized effective dimension $\hat d_{\mathrm{eff}}$ vs $D/M$, one curve per fixed (M, Dloc) architecture, at the fixed target $\mathrm{tr}\,S^4=$ `purity0`.

In [ ]:
fs = 28
figsize = (8,4)

plt.rcParams['xtick.labelsize'] = fs
plt.rcParams['ytick.labelsize'] = fs
plt.rcParams["figure.figsize"] = figsize
plt.rcParams['text.usetex'] = True

fig = plt.figure(figsize=figsize)
ax = fig.add_subplot(111)

clrs = ['tab:blue', 'tab:orange']
for i in range(no_exps):
    dict_exp = dict_exps[i]
    no_params = dict_exp['no_params']
    local_dim_param = dict_exp['local_dim_param']
    dim_in_all = dict_exp['dim_in_all']
    norm_eff_dim_all = dict_exp['norm_eff_dim_all']
    purity_S_all = numpy.asarray(dict_exp['purity_S_all'])
    purity_rounded = round(numpy.mean(purity_S_all), 2)
    print('S purity: ', purity_rounded, ' with std. ', np.std(purity_S_all))
    lbl = r'$M=' + str(no_params) + ',\,' + r'\tilde{d}=' + str(local_dim_param) + ',\,\mathrm{tr}\,S^4=' + str(purity_rounded) + '$'
    x = np.asarray(dim_in_all) / no_params
    y = np.asarray(norm_eff_dim_all)
    ax.plot(x, y, '.', color=clrs[i], markersize=12, label=lbl)

#ax.legend(fontsize=22, loc='lower left')
ax.legend(fontsize=22)
ax.set_xlabel(r'$D/M$', fontsize=fs)
ax.set_ylabel(r'$\hat{d}_{\mathrm{eff}}$', fontsize=fs)
ax.set_xscale('log')
#ax.set_yscale('log')
#ax.set_ylim([0.725, 0.86])

fig.savefig('EffDimScaling_vs_D_TN__chi_120.png', bbox_inches='tight')
fig.savefig('EffDimScaling_vs_D_TN__chi_120.pdf', bbox_inches='tight')